# 02 · One-stage anchor-based: RetinaNet (ResNet-50-FPN v2, torchvision)
Same protocol as 01 (frame step, epochs, optimizer, evaluator).

Kaggle: Accelerator **GPU**, **Internet on**. Run with *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <- change once: your public repo with src/common.py
FAST_DEV = os.environ.get('FAST_DEV') == '1'                # tiny smoke test on a fake dataset; keep off on Kaggle

if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
sys.path.insert(0, next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists()))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools'], check=True)

import math
import pickle
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.models.detection.retinanet as tv_retina
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import sigmoid_focal_loss
from common import *

seed_everything()
ROOT = find_data_root()
TMP = Path('/tmp/hw2') if not FAST_DEV else Path('work_dev')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
FIG = Path('images'); FIG.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

## 1. Protocol (identical to 01)

In [ ]:
STEP = 5
EPOCHS = 4 if not FAST_DEV else 1
BATCH = 4 if not FAST_DEV else 2
LR = 0.005            # RetinaNet is less stable than Faster R-CNN at the same LR
PRETRAINED = not FAST_DEV

yaml_path = prepare_training_copy(ROOT, TMP / 'data', step_train=STEP, step_val=STEP)
manifest = TMP / 'data' / 'manifest.csv'
train_ds = build_torch_dataset(manifest, 'train', train=True)
val_frames = list_frames(ROOT / 'val', STEP)
evaluator = CocoEvaluator(ROOT, val_frames)
print('train images:', len(train_ds), '| val frames:', len(val_frames))

## 2. Anchors grounded in the data
Default torchvision anchors: base sizes 32–512 px on P3–P7 (3 scales × ratios h/w 0.5/1/2).
Below: object sizes at the model input scale (long side ≈ 1333) and how well each anchor set covers them
(share of GT boxes whose best anchor has IoU ≥ 0.5, i.e. that get a "proper" positive).

In [ ]:
man = pd.read_csv(manifest)
wh = []
for r in man[man.split == 'train'].sample(min(1500, (man.split == 'train').sum()), random_state=SEED).itertuples():
    im = cv2.imread(r.image); h, w = im.shape[:2]
    b, _ = read_yolo_labels(Path(r.image.replace('/images/', '/labels/')).with_suffix('.txt'), w, h)
    s = min(800 / min(h, w), 1333 / max(h, w))  # torchvision GeneralizedRCNNTransform scale
    wh.append((b[:, 2:] - b[:, :2]) * s)
wh = np.concatenate(wh)
size = np.sqrt(wh.prod(1)); ratio = wh[:, 1] / wh[:, 0]
print('size at input scale, quantiles 5/25/50/75/95%:', np.quantile(size, [.05, .25, .5, .75, .95]).round(1))
print('ratio h/w quantiles 10/50/90%:', np.quantile(ratio, [.1, .5, .9]).round(2))


def anchor_set(base_sizes, ratios):
    sizes = tuple((x, x * 2 ** (1 / 3), x * 2 ** (2 / 3)) for x in base_sizes)
    return sizes, (tuple(ratios),) * len(sizes)


def anchor_coverage(sizes, ratios, thr=0.5):
    """Best IoU between a GT shape and any anchor shape (both centred) — upper bound of matching quality."""
    shapes = []
    for lvl_sizes, lvl_ratios in zip(sizes, ratios):
        for s in lvl_sizes:
            for r in lvl_ratios:
                shapes.append((s / math.sqrt(r), s * math.sqrt(r)))  # (w, h) with h/w = r
    A = np.array(shapes)
    inter = np.minimum(wh[:, None, 0], A[None, :, 0]) * np.minimum(wh[:, None, 1], A[None, :, 1])
    iou = inter / (wh.prod(1)[:, None] + A.prod(1)[None] - inter)
    best = iou.max(1)
    return round(float((best >= thr).mean()), 3), round(float(np.median(best)), 3)


DATA_RATIOS = tuple(np.round(np.quantile(ratio, [.15, .5, .85]), 2))  # 3 ratios -> pretrained head shape is kept
ANCHORS = {
    'default (32-512, 0.5/1/2)': anchor_set([32, 64, 128, 256, 512], (0.5, 1.0, 2.0)),
    'small (16-256, 0.5/1/2)': anchor_set([16, 32, 64, 128, 256], (0.5, 1.0, 2.0)),
    f'small + data ratios {DATA_RATIOS}': anchor_set([16, 32, 64, 128, 256], DATA_RATIOS),
    f'tiny (8-128) + data ratios': anchor_set([8, 16, 32, 64, 128], DATA_RATIOS),
}
cov = pd.DataFrame([(k, *anchor_coverage(*v)) for k, v in ANCHORS.items()], columns=['anchors', 'share_GT_IoU>=0.5', 'median_best_IoU'])
print(cov.to_string(index=False))

## 3. Experiments (incremental)
* `base` – default anchors, focal loss α=0.25 γ=2, IoU thresholds fg 0.5 / bg 0.4.
* `anchors` – data-driven anchors (sizes / ratios from section 2).
* `anchors_iou` – + lower assignment thresholds (fg 0.4 / bg 0.3): tiny boxes rarely reach IoU 0.5 with any anchor (optional run).
* `anchors_focal` – + lower IoU thresholds + focal loss γ=1.5, α=0.5 (loss modification).

In [ ]:
SMALL = ANCHORS[f'small + data ratios {DATA_RATIOS}']
EXPERIMENTS = {
    'retina_base':          dict(),
    'retina_anchors':       dict(anchors=SMALL),
    'retina_anchors_iou':   dict(anchors=SMALL, fg_iou_thresh=0.4, bg_iou_thresh=0.3),
    'retina_anchors_focal': dict(anchors=SMALL, fg_iou_thresh=0.4, bg_iou_thresh=0.3, focal=(0.5, 1.5)),
}
RUN = ['retina_base', 'retina_anchors', 'retina_anchors_focal']  # add 'retina_anchors_iou' if GPU time allows
if FAST_DEV:
    RUN = RUN[:2]


def build_retina(cfg):
    cfg = dict(cfg)
    anchors = cfg.pop('anchors', None)
    alpha, gamma = cfg.pop('focal', (0.25, 2.0))
    # torchvision calls `sigmoid_focal_loss` from the retinanet module -> patch it to change alpha / gamma
    tv_retina.sigmoid_focal_loss = partial(sigmoid_focal_loss, alpha=alpha, gamma=gamma)
    model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1 if PRETRAINED else None,
                                      weights_backbone=None, detections_per_img=500, score_thresh=0.001,
                                      topk_candidates=1000, **(dict(min_size=320, max_size=512) if FAST_DEV else {}), **cfg)
    head = model.head.classification_head
    na = head.num_anchors
    head.num_classes = NC + 1                     # keep the pretrained conv tower, replace only the last layer
    head.cls_logits = torch.nn.Conv2d(256, na * (NC + 1), 3, padding=1)
    torch.nn.init.normal_(head.cls_logits.weight, std=0.01)
    torch.nn.init.constant_(head.cls_logits.bias, -math.log((1 - 0.01) / 0.01))
    if anchors:
        model.anchor_generator = AnchorGenerator(*anchors)
        assert model.anchor_generator.num_anchors_per_location()[0] == na
    return model

## 4. Train and evaluate

In [ ]:
histories, results = {}, {}
for name in RUN:
    print(f'\n===== {name} =====')
    seed_everything()
    model = build_retina(EXPERIMENTS[name])
    histories[name] = train_torchvision(model, train_ds, evaluator, val_frames, DEVICE, epochs=EPOCHS, lr=LR,
                                        batch=BATCH, out_dir=TMP / 'runs' / name)
    preds, fps = predict_torchvision(model, val_frames, DEVICE)
    m = evaluator.evaluate(preds)
    results[name] = preds
    log_result(dict(model=name, family='one-stage anchor', epochs=EPOCHS, step=STEP, fps=fps, **m), OUT / 'metrics.csv')
    pd.DataFrame(histories[name]).to_csv(OUT / f'history_{name}.csv', index=False)
    pickle.dump(preds, open(TMP / f'val_preds_{name}.pkl', 'wb'))
    print(name, {k: m[k] for k in ['mAP', 'AP50', 'AP_small']}, 'fps', fps)
    del model; torch.cuda.empty_cache()
tv_retina.sigmoid_focal_loss = sigmoid_focal_loss  # restore

## 5. Results and comparison with the two-stage detector

In [ ]:
tab = pd.read_csv(OUT / 'metrics.csv')
cols = ['model', 'family', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AR500', 'fps']
print(tab[cols].to_string(index=False))  # includes Faster R-CNN rows if results/metrics.csv from 01 is present

plt.figure(figsize=(6, 3.5))
for name, h in histories.items():
    h = pd.DataFrame(h); plt.plot(h.epoch, h.mAP, marker='o', label=name)
plt.xlabel('epoch'); plt.ylabel('val mAP'); plt.legend(); plt.title('RetinaNet: val mAP per epoch')
plt.tight_layout(); plt.savefig(FIG / 'retina_curves.png', dpi=120); plt.show()

BEST = max(results, key=lambda n: evaluator.evaluate(results[n])['mAP'])
print('best one-stage anchor-based run:', BEST)
print(evaluator.evaluate_by_size(results[BEST]).to_string(index=False))

## 6. No proposal stage: foreground / background imbalance
RetinaNet classifies every anchor directly. Count anchors per image vs. anchors assigned as positives.

In [ ]:
model = build_retina(EXPERIMENTS[BEST]); model.eval()
img = torch.zeros(3, 1080, 1920)
images, _ = model.transform([img])
feats = list(model.backbone(images.tensors).values())
n_anchors = sum(f.shape[-2] * f.shape[-1] for f in feats) * model.anchor_generator.num_anchors_per_location()[0]
anchors = model.anchor_generator(images, feats)[0]
pos = []
for r in man[man.split == 'train'].sample(min(100, (man.split == 'train').sum()), random_state=SEED).itertuples():
    im = cv2.imread(r.image); h, w = im.shape[:2]
    b, _ = read_yolo_labels(Path(r.image.replace('/images/', '/labels/')).with_suffix('.txt'), w, h)
    if len(b) == 0:
        continue
    s = images.image_sizes[0][1] / w
    match = model.proposal_matcher(torchvision.ops.box_iou(torch.as_tensor(b * s, dtype=torch.float32), anchors))
    pos.append(int((match >= 0).sum()))
print(f'anchors per 1920x1080 frame: {n_anchors:,} | mean positives: {np.mean(pos):.0f} | ratio 1:{n_anchors / np.mean(pos):.0f}')
del model

## 7. Notes for the report
*(your analysis: anchor choice vs object sizes, effect of IoU thresholds / focal loss,
what is lost without the proposal stage and how focal loss / assignment compensate, comparison with tuned Faster R-CNN)*